In [1]:
import pandas as pd
import numpy as np

In [2]:
train_data = pd.read_csv("sample_loans_train.csv")

train_data.head()

,index,target,CreditScore,FirstPaymentDate,FirstTimeHomebuyerFlag,MaturityDate,MSA,MI_Pct,NumberOfUnits,OccupancyStatus,...,12_MonthlyReportingPeriod,12_RemainingMonthsToLegalMaturity,13_CurrentActualUPB,13_CurrentInterestRate,13_CurrentNonInterestBearingUPB,13_EstimatedLTV,13_InterestBearingUPB,13_LoanAge,13_MonthlyReportingPeriod,13_RemainingMonthsToLegalMaturity
0,0,0,747,202403,N,205402,NaN,0,1,P,...,202502,348,81190.87,8.000,0,999,81190.87,13,202503,347
1,1,0,659,202403,N,205402,NaN,0,1,P,...,202502,348,301932.72,7.875,0,64,301932.72,13,202503,347
2,3,0,775,202403,N,205402,46540.0,0,2,I,...,202502,348,152044.42,7.625,0,999,152044.42,13,202503,347
3,4,0,815,202403,Y,205402,NaN,25,1,P,...,202502,348,50045.91,7.250,0,70,50045.91,13,202503,347
4,6,0,772,202403,N,205402,10900.0,0,1,P,...,202502,348,153521.39,7.750,0,63,153521.39,13,202503,347


In [3]:
valid_data = pd.read_csv("sample_loans_valid.csv")

valid_data.head()

,index,target,CreditScore,FirstPaymentDate,FirstTimeHomebuyerFlag,MaturityDate,MSA,MI_Pct,NumberOfUnits,OccupancyStatus,...,12_MonthlyReportingPeriod,12_RemainingMonthsToLegalMaturity,13_CurrentActualUPB,13_CurrentInterestRate,13_CurrentNonInterestBearingUPB,13_EstimatedLTV,13_InterestBearingUPB,13_LoanAge,13_MonthlyReportingPeriod,13_RemainingMonthsToLegalMaturity
0,2,0,784,202403,N,205402,46540.0,0,1,P,...,202502,348,123552.68,6.750,0.0,69,123552.68,13,202503,347
1,11,0,786,202403,N,205402,15764.0,25,1,P,...,202502,348,605030.78,7.000,0.0,71,605030.78,13,202503,347
2,22,0,739,202403,N,205402,36540.0,0,1,P,...,202502,348,158234.13,7.000,0.0,21,158234.13,13,202503,347
3,38,0,773,202403,N,205402,NaN,0,1,P,...,202502,348,155843.09,6.875,0.0,58,155843.09,13,202503,347
4,39,0,736,202403,Y,205402,NaN,25,1,P,...,202502,348,66226.11,7.250,0.0,88,66226.11,13,202503,347


In [4]:
train_data.isnull().mean().sort_values(ascending=False).head(10)

ReliefRefinanceIndicator            1.000000
PreHARP_Flag                        1.000000
SuperConformingFlag                 0.989018
MSA                                 0.142763
index                               0.000000
7_RemainingMonthsToLegalMaturity    0.000000
8_CurrentActualUPB                  0.000000
8_CurrentInterestRate               0.000000
8_CurrentNonInterestBearingUPB      0.000000
8_EstimatedLTV                      0.000000
dtype: float64

In [5]:
train_data = train_data.drop(columns=['ReliefRefinanceIndicator','PreHARP_Flag'])
valid_data = valid_data.drop(columns=['ReliefRefinanceIndicator','PreHARP_Flag'])

In [6]:
print((train_data['CreditScore'] == 9999).sum())
print((train_data['MI_Pct'] == 999).sum())
print((train_data['OriginalDTI'] == 999).sum())
print((train_data['OriginalLTV'] == 999).sum())

for col in train_data.columns:
    if '_' in col:
        sp = col.split('_')
        if sp[1] == 'EstimatedLTV':
            print(col, ":", (train_data[col] == 999).sum())

2
0
0
0
0_EstimatedLTV : 608
1_EstimatedLTV : 474
2_EstimatedLTV : 476
3_EstimatedLTV : 450
4_EstimatedLTV : 454
5_EstimatedLTV : 460
6_EstimatedLTV : 460
7_EstimatedLTV : 454
8_EstimatedLTV : 448
9_EstimatedLTV : 445
10_EstimatedLTV : 443
11_EstimatedLTV : 450
12_EstimatedLTV : 445
13_EstimatedLTV : 493


In [7]:
print((valid_data['CreditScore'] == 9999).sum())
print((valid_data['MI_Pct'] == 999).sum())
print((valid_data['OriginalDTI'] == 999).sum())
print((valid_data['OriginalLTV'] == 999).sum())

for col in valid_data.columns:
    if '_' in col:
        sp = col.split('_')
        if sp[1] == 'EstimatedLTV':
            print(col, ":", (valid_data[col] == 999).sum())

0
0
0
0
0_EstimatedLTV : 103
1_EstimatedLTV : 85
2_EstimatedLTV : 86
3_EstimatedLTV : 83
4_EstimatedLTV : 81
5_EstimatedLTV : 82
6_EstimatedLTV : 82
7_EstimatedLTV : 81
8_EstimatedLTV : 80
9_EstimatedLTV : 81
10_EstimatedLTV : 78
11_EstimatedLTV : 78
12_EstimatedLTV : 77
13_EstimatedLTV : 88


In [8]:
def spec_num(df: pd.DataFrame):
    
    df['CreditScoreMissFLag'] = (df['CreditScore'] == 9999).astype(int)
    med = df.loc[df['CreditScore'] != 9999, 'CreditScore'].median()
    df['CreditScore'] = df['CreditScore'].replace(9999, med)
   
    df['MI_PctMissFlag'] = (df['MI_Pct'] == 999).astype(int)
    med = df.loc[df['MI_Pct'] != 999, 'MI_Pct'].median()
    df['MI_Pct'] = df['MI_Pct'].replace(999, med)

    df['OriginalDTIMissFlag'] = (df['OriginalDTI'] == 999).astype(int)
    med = df.loc[df['OriginalDTI'] != 999, 'OriginalDTI'].median()
    df['OriginialDTI'] = df['OriginalDTI'].replace(999, med)
    
    df['OriginalLTVMissFlag'] = (df['OriginalLTV'] == 999).astype(int)
    med = df.loc[df['OriginalLTV'] != 999, 'OriginalLTV'].median()
    df['OriginialLTV'] = df['OriginalLTV'].replace(999, med)

    return df

In [9]:
train_data = spec_num(train_data)
valid_data = spec_num(valid_data)

print((train_data['CreditScore'] == 9999).sum())
print((train_data['MI_Pct'] == 999).sum())
print((train_data['OriginalDTI'] == 999).sum())
print((train_data['OriginalLTV'] == 999).sum())

print((valid_data['CreditScore'] == 9999).sum())
print((valid_data['MI_Pct'] == 999).sum())
print((valid_data['OriginalDTI'] == 999).sum())
print((valid_data['OriginalLTV'] == 999).sum())

0
0
0
0
0
0
0
0


In [10]:
def updateLTV(row, ltv, upb):
    
    if row[ltv].isna().all():
        reconstructed = (row[upb].values / row['OriginalPropertyValue']) * 100
        return pd.Series(reconstructed, index=ltv)
    else:
        filled = row[ltv].copy()
        filled = filled.fillna(method = 'ffill').fillna(method = 'bfill')
        filled = filled.fillna(filled.median())
        return filled

In [11]:
def spec_dyn_LTV(df: pd.DataFrame):
    
    ltv_col = []
    upb_col = []
    
    for c in df.columns:
        if 'EstimatedLTV' in c:
            ltv_col.append(c)
        
        if 'CurrentActualUPB' in c:
            upb_col.append(c)
    
    df[ltv_col] = df[ltv_col].replace(999, np.nan)

    df['EstimatedLTV_missing_count'] = df[ltv_col].isna().sum(axis=1)
    df['EstimatedLTV_all_MissFlag'] = (df['EstimatedLTV_missing_count'] == len(ltv_col)).astype(int)
    df['OriginalPropertyValue'] = df['OriginalUPB']/(df['OriginalLTV']/100)

    df[ltv_col] = df.apply(updateLTV, axis = 1, ltv = ltv_col, upb = upb_col)

    df = df.drop(columns = ['OriginalPropertyValue', 'EstimatedLTV_missing_count'])

    return df

In [12]:
train_data = spec_dyn_LTV(train_data)
valid_data = spec_dyn_LTV(valid_data)

/var/folders/hd/1qp006zn48sdflzfg6_25rtr0000gn/T/ipykernel_63778/1004315372.py:8: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  filled = filled.fillna(method = 'ffill').fillna(method = 'bfill')
/var/folders/hd/1qp006zn48sdflzfg6_25rtr0000gn/T/ipykernel_63778/1004315372.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  filled = filled.fillna(method = 'ffill').fillna(method = 'bfill')
/var/folders/hd/1qp006zn48sdflzfg6_25rtr0000gn/T/ipykernel_63778/1004315372.py:8: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  filled = filled.fillna(method = 'ffill').fillna(method = 'bfill')
/var/folders/hd/1qp0

In [13]:
for col in train_data.columns:
    if '_' in col:
        sp = col.split('_')
        if sp[1] == 'EstimatedLTV':
            print(col, ":", (train_data[col] == 999).sum())

for col in valid_data.columns:
    if '_' in col:
        sp = col.split('_')
        if sp[1] == 'EstimatedLTV':
            print(col, ":", (valid_data[col] == 999).sum())

0_EstimatedLTV : 0
1_EstimatedLTV : 0
2_EstimatedLTV : 0
3_EstimatedLTV : 0
4_EstimatedLTV : 0
5_EstimatedLTV : 0
6_EstimatedLTV : 0
7_EstimatedLTV : 0
8_EstimatedLTV : 0
9_EstimatedLTV : 0
10_EstimatedLTV : 0
11_EstimatedLTV : 0
12_EstimatedLTV : 0
13_EstimatedLTV : 0
0_EstimatedLTV : 0
1_EstimatedLTV : 0
2_EstimatedLTV : 0
3_EstimatedLTV : 0
4_EstimatedLTV : 0
5_EstimatedLTV : 0
6_EstimatedLTV : 0
7_EstimatedLTV : 0
8_EstimatedLTV : 0
9_EstimatedLTV : 0
10_EstimatedLTV : 0
11_EstimatedLTV : 0
12_EstimatedLTV : 0
13_EstimatedLTV : 0


In [14]:
cat_col = ['FirstTimeHomebuyerFlag', 'MSA', 'NumberOfUnits', 'OccupancyStatus', 'Channel', 'PPM_Flag', 'ProductType', 'PropertyState', 
           'PropertyType', 'PostalCode', 'LoanPurpose', 'SellerName', 'ServicerName', 'SuperConformingFlag', 'ProgramIndicator', 
           'PropertyValMethod', 'InterestOnlyFlag', 'BalloonIndicator']

for c in cat_col:
    print(c, ":", train_data[c].nunique())

for c in cat_col:
    print(c, ":", valid_data[c].nunique())

FirstTimeHomebuyerFlag : 2
MSA : 382
NumberOfUnits : 4
OccupancyStatus : 3
Channel : 3
PPM_Flag : 1
ProductType : 1
PropertyState : 54
PropertyType : 5
PostalCode : 758
LoanPurpose : 3
SellerName : 20
ServicerName : 19
SuperConformingFlag : 1
ProgramIndicator : 3
PropertyValMethod : 5
InterestOnlyFlag : 1
BalloonIndicator : 3
FirstTimeHomebuyerFlag : 2
MSA : 261
NumberOfUnits : 4
OccupancyStatus : 3
Channel : 3
PPM_Flag : 1
ProductType : 1
PropertyState : 52
PropertyType : 4
PostalCode : 473
LoanPurpose : 3
SellerName : 20
ServicerName : 18
SuperConformingFlag : 1
ProgramIndicator : 3
PropertyValMethod : 4
InterestOnlyFlag : 1
BalloonIndicator : 3


In [15]:
def Freq_Endcode(df: pd.DataFrame, col):
    ch = df[col].value_counts(normalize=False).to_dict()
    en = col + "Encoded"
    df[en] = df[col].map(ch)
    name = col + "MissFlag"
    df[name] = df[col].isna().astype(int)
    df[en] = df[en].fillna(0)

In [16]:
fen = ['MSA', 'PropertyState', 'SellerName', 'ServicerName']

for c in fen:
    Freq_Endcode(train_data, c)

for c in fen:
    Freq_Endcode(valid_data, c)

In [17]:
def is_num(s):
    try:
        int(s)
        return True
    except ValueError:
        return False

In [18]:
def sep_dyn_sta(df: pd.DataFrame):
    dyn_col = []
    stat_col = []

    for c in df.columns:
        arr = c.split('_')
        if is_num(arr[0]):
            dyn_col.append(c)

    for c in df.columns:
        if c not in dyn_col:
            stat_col.append(c)

    return dyn_col, stat_col

In [19]:
def reshape_data(df: pd.DataFrame):
    dyn_col, stat_col = sep_dyn_sta(df)

    dynamic_feature_names = set([c.split("_")[1] for c in dyn_col])
    dynamic_feature_names

    long_dfs = []
    
    for feat in dynamic_feature_names:
        feat_cols = [c for c in df.columns if c.endswith("_" + feat)]
        
        df_feat_long = df.melt(
            id_vars=stat_col, 
            value_vars=feat_cols,
            var_name="month_feat", 
            value_name=feat
        )
        
        df_feat_long["month"] = df_feat_long["month_feat"].str.extract(r"(\d+)_")[0].astype(int)
        df_feat_long = df_feat_long.drop(columns=["month_feat"])
        
        long_dfs.append(df_feat_long)

    df_long = long_dfs[0]
    
    for extra in long_dfs[1:]:
        df_long = df_long.merge(extra, on=stat_col + ["month"], how="outer")

    return df_long

In [20]:
train_data_long = reshape_data(train_data)

train_data_long.columns

Index(['index', 'target', 'CreditScore', 'FirstPaymentDate',
       'FirstTimeHomebuyerFlag', 'MaturityDate', 'MSA', 'MI_Pct',
       'NumberOfUnits', 'OccupancyStatus', 'OriginalCLTV', 'OriginalDTI',
       'OriginalUPB', 'OriginalLTV', 'OriginalInterestRate', 'Channel',
       'PPM_Flag', 'ProductType', 'PropertyState', 'PropertyType',
       'PostalCode', 'LoanPurpose', 'OriginalLoanTerm', 'NumberOfBorrowers',
       'SellerName', 'ServicerName', 'SuperConformingFlag', 'ProgramIndicator',
       'PropertyValMethod', 'InterestOnlyFlag', 'BalloonIndicator',
       'CreditScoreMissFLag', 'MI_PctMissFlag', 'OriginalDTIMissFlag',
       'OriginialDTI', 'OriginalLTVMissFlag', 'OriginialLTV',
       'EstimatedLTV_all_MissFlag', 'MSAEncoded', 'MSAMissFlag',
       'PropertyStateEncoded', 'PropertyStateMissFlag', 'SellerNameEncoded',
       'SellerNameMissFlag', 'ServicerNameEncoded', 'ServicerNameMissFlag',
       'CurrentNonInterestBearingUPB', 'month', 'LoanAge',
       'RemainingMonthsTo

In [21]:
valid_data_long = reshape_data(valid_data)

valid_data_long.columns

Index(['index', 'target', 'CreditScore', 'FirstPaymentDate',
       'FirstTimeHomebuyerFlag', 'MaturityDate', 'MSA', 'MI_Pct',
       'NumberOfUnits', 'OccupancyStatus', 'OriginalCLTV', 'OriginalDTI',
       'OriginalUPB', 'OriginalLTV', 'OriginalInterestRate', 'Channel',
       'PPM_Flag', 'ProductType', 'PropertyState', 'PropertyType',
       'PostalCode', 'LoanPurpose', 'OriginalLoanTerm', 'NumberOfBorrowers',
       'SellerName', 'ServicerName', 'SuperConformingFlag', 'ProgramIndicator',
       'PropertyValMethod', 'InterestOnlyFlag', 'BalloonIndicator',
       'CreditScoreMissFLag', 'MI_PctMissFlag', 'OriginalDTIMissFlag',
       'OriginialDTI', 'OriginalLTVMissFlag', 'OriginialLTV',
       'EstimatedLTV_all_MissFlag', 'MSAEncoded', 'MSAMissFlag',
       'PropertyStateEncoded', 'PropertyStateMissFlag', 'SellerNameEncoded',
       'SellerNameMissFlag', 'ServicerNameEncoded', 'ServicerNameMissFlag',
       'CurrentNonInterestBearingUPB', 'month', 'LoanAge',
       'RemainingMonthsTo

In [22]:
extra_col = ['FirstPaymentDate', 'MaturityDate', 'MSA', 'PropertyState', 'PostalCode','SellerName', 'ServicerName', 
             'MonthlyReportingPeriod', 'month']

train_data_long = train_data_long.drop(columns = extra_col)
valid_data_long = valid_data_long.drop(columns = extra_col)

In [23]:
train_data_long.isna().sum()

index                                 0
target                                0
CreditScore                           0
FirstTimeHomebuyerFlag                0
MI_Pct                                0
NumberOfUnits                         0
OccupancyStatus                       0
OriginalCLTV                          0
OriginalDTI                           0
OriginalUPB                           0
OriginalLTV                           0
OriginalInterestRate                  0
Channel                               0
PPM_Flag                              0
ProductType                           0
PropertyType                          0
LoanPurpose                           0
OriginalLoanTerm                      0
NumberOfBorrowers                     0
SuperConformingFlag               84476
ProgramIndicator                      0
PropertyValMethod                     0
InterestOnlyFlag                      0
BalloonIndicator                      0
CreditScoreMissFLag                   0


In [24]:
valid_data_long.isna().sum()

index                                 0
target                                0
CreditScore                           0
FirstTimeHomebuyerFlag                0
MI_Pct                                0
NumberOfUnits                         0
OccupancyStatus                       0
OriginalCLTV                          0
OriginalDTI                           0
OriginalUPB                           0
OriginalLTV                           0
OriginalInterestRate                  0
Channel                               0
PPM_Flag                              0
ProductType                           0
PropertyType                          0
LoanPurpose                           0
OriginalLoanTerm                      0
NumberOfBorrowers                     0
SuperConformingFlag               14924
ProgramIndicator                      0
PropertyValMethod                     0
InterestOnlyFlag                      0
BalloonIndicator                      0
CreditScoreMissFLag                   0


In [25]:
c_col = ['FirstTimeHomebuyerFlag', 'NumberOfUnits', 'OccupancyStatus', 'Channel', 'PPM_Flag', 'ProductType', 'PropertyType', 
         'LoanPurpose', 'SuperConformingFlag', 'ProgramIndicator', 'PropertyValMethod', 'InterestOnlyFlag', 'BalloonIndicator']

n_col = []

for c in train_data_long.columns:
    if c not in c_col:
        n_col.append(c)

In [26]:
for c in c_col:
    print(c, ":", train_data_long[c].unique())

for c in c_col:
    print(c, ":", valid_data_long[c].unique())

FirstTimeHomebuyerFlag : ['N' 'Y']
NumberOfUnits : [1 2 4 3]
OccupancyStatus : ['P' 'I' 'S']
Channel : ['R' 'C' 'B']
PPM_Flag : ['N']
ProductType : ['FRM']
PropertyType : ['MH' 'SF' 'PU' 'CO' 'CP']
LoanPurpose : ['P' 'C' 'N']
SuperConformingFlag : [nan 'Y']
ProgramIndicator : ['9' 'H' 'F']
PropertyValMethod : [2 1 3 4 9]
InterestOnlyFlag : ['N']
BalloonIndicator : ['7' 'N' 'Y']
FirstTimeHomebuyerFlag : ['N' 'Y']
NumberOfUnits : [1 2 3 4]
OccupancyStatus : ['P' 'S' 'I']
Channel : ['R' 'C' 'B']
PPM_Flag : ['N']
ProductType : ['FRM']
PropertyType : ['SF' 'PU' 'MH' 'CO']
LoanPurpose : ['C' 'P' 'N']
SuperConformingFlag : [nan 'Y']
ProgramIndicator : ['9' 'H' 'F']
PropertyValMethod : [2 1 4 3]
InterestOnlyFlag : ['N']
BalloonIndicator : ['7' 'N' 'Y']


In [27]:
train_data_long['SuperConformingFlag'] = train_data_long['SuperConformingFlag'].replace(np.nan, 'N')
valid_data_long['SuperConformingFlag'] = valid_data_long['SuperConformingFlag'].replace(np.nan, 'N')

In [28]:
train_data_long.isna().sum()

index                             0
target                            0
CreditScore                       0
FirstTimeHomebuyerFlag            0
MI_Pct                            0
NumberOfUnits                     0
OccupancyStatus                   0
OriginalCLTV                      0
OriginalDTI                       0
OriginalUPB                       0
OriginalLTV                       0
OriginalInterestRate              0
Channel                           0
PPM_Flag                          0
ProductType                       0
PropertyType                      0
LoanPurpose                       0
OriginalLoanTerm                  0
NumberOfBorrowers                 0
SuperConformingFlag               0
ProgramIndicator                  0
PropertyValMethod                 0
InterestOnlyFlag                  0
BalloonIndicator                  0
CreditScoreMissFLag               0
MI_PctMissFlag                    0
OriginalDTIMissFlag               0
OriginialDTI                

In [29]:
valid_data_long.isna().sum()

index                             0
target                            0
CreditScore                       0
FirstTimeHomebuyerFlag            0
MI_Pct                            0
NumberOfUnits                     0
OccupancyStatus                   0
OriginalCLTV                      0
OriginalDTI                       0
OriginalUPB                       0
OriginalLTV                       0
OriginalInterestRate              0
Channel                           0
PPM_Flag                          0
ProductType                       0
PropertyType                      0
LoanPurpose                       0
OriginalLoanTerm                  0
NumberOfBorrowers                 0
SuperConformingFlag               0
ProgramIndicator                  0
PropertyValMethod                 0
InterestOnlyFlag                  0
BalloonIndicator                  0
CreditScoreMissFLag               0
MI_PctMissFlag                    0
OriginalDTIMissFlag               0
OriginialDTI                

In [30]:
for c in c_col:
    print(c, ":", train_data_long[c].unique())

for c in c_col:
    print(c, ":", valid_data_long[c].unique())

FirstTimeHomebuyerFlag : ['N' 'Y']
NumberOfUnits : [1 2 4 3]
OccupancyStatus : ['P' 'I' 'S']
Channel : ['R' 'C' 'B']
PPM_Flag : ['N']
ProductType : ['FRM']
PropertyType : ['MH' 'SF' 'PU' 'CO' 'CP']
LoanPurpose : ['P' 'C' 'N']
SuperConformingFlag : ['N' 'Y']
ProgramIndicator : ['9' 'H' 'F']
PropertyValMethod : [2 1 3 4 9]
InterestOnlyFlag : ['N']
BalloonIndicator : ['7' 'N' 'Y']
FirstTimeHomebuyerFlag : ['N' 'Y']
NumberOfUnits : [1 2 3 4]
OccupancyStatus : ['P' 'S' 'I']
Channel : ['R' 'C' 'B']
PPM_Flag : ['N']
ProductType : ['FRM']
PropertyType : ['SF' 'PU' 'MH' 'CO']
LoanPurpose : ['C' 'P' 'N']
SuperConformingFlag : ['N' 'Y']
ProgramIndicator : ['9' 'H' 'F']
PropertyValMethod : [2 1 4 3]
InterestOnlyFlag : ['N']
BalloonIndicator : ['7' 'N' 'Y']


In [31]:
yn_col = []
allowed = ['y', 'n']

for c in c_col:
    d = train_data_long[c].astype(str)
    if d.str.lower().isin(allowed).all():
        yn_col.append(c)

In [32]:
def bin_encode(df: pd.DataFrame, yn_col):
    
    for c in yn_col:
        df[c] = df[c].replace({'Y':1, 'N':0})

    return df

In [33]:
train_data_long = bin_encode(train_data_long, yn_col)
valid_data_long = bin_encode(valid_data_long, yn_col)

/var/folders/hd/1qp006zn48sdflzfg6_25rtr0000gn/T/ipykernel_63778/3607371288.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[c] = df[c].replace({'Y':1, 'N':0})


In [34]:
for c in yn_col:
    c_col.remove(c)
    n_col.append(c)

In [35]:
print(len(train_data_long.columns))
print(len(c_col) + len(n_col))

46
46


In [36]:
print(len(valid_data_long.columns))
print(len(c_col) + len(n_col))

46
46


In [37]:
c_col

['NumberOfUnits',
 'OccupancyStatus',
 'Channel',
 'ProductType',
 'PropertyType',
 'LoanPurpose',
 'ProgramIndicator',
 'PropertyValMethod',
 'BalloonIndicator']

In [38]:
n_col

['index',
 'target',
 'CreditScore',
 'MI_Pct',
 'OriginalCLTV',
 'OriginalDTI',
 'OriginalUPB',
 'OriginalLTV',
 'OriginalInterestRate',
 'OriginalLoanTerm',
 'NumberOfBorrowers',
 'CreditScoreMissFLag',
 'MI_PctMissFlag',
 'OriginalDTIMissFlag',
 'OriginialDTI',
 'OriginalLTVMissFlag',
 'OriginialLTV',
 'EstimatedLTV_all_MissFlag',
 'MSAEncoded',
 'MSAMissFlag',
 'PropertyStateEncoded',
 'PropertyStateMissFlag',
 'SellerNameEncoded',
 'SellerNameMissFlag',
 'ServicerNameEncoded',
 'ServicerNameMissFlag',
 'CurrentNonInterestBearingUPB',
 'LoanAge',
 'RemainingMonthsToLegalMaturity',
 'CurrentInterestRate',
 'EstimatedLTV',
 'InterestBearingUPB',
 'CurrentActualUPB',
 'FirstTimeHomebuyerFlag',
 'PPM_Flag',
 'SuperConformingFlag',
 'InterestOnlyFlag']

In [39]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, roc_auc_score

X_train = train_data_long[n_col + c_col]
X_valid = valid_data_long[n_col + c_col]
Y_valid = valid_data[['index', 'target']]

In [40]:
X_train = X_train.drop(columns = ["target", "index"])
X_valid = X_valid.drop(columns = ["target", "index"])

n_col.remove("target")
n_col.remove("index")

In [51]:
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), c_col),
        ("num", StandardScaler(with_mean=False), n_col),
    ],
    sparse_threshold=1.0, 
)

model = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination="auto",
    random_state=42,
    n_jobs=-1,
)

pipe = Pipeline([("prep", preprocess), ("clf", model)])
pipe.fit(X_train)

,steps,"[('prep', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,1.0
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [52]:
X_valid_scaled = pipe["prep"].transform(X_valid)
scores = pipe["clf"].score_samples(X_valid_scaled)
raw_anom = -scores

min_v, max_v = np.min(raw_anom), np.max(raw_anom)
anom_score = (raw_anom - min_v) / (max_v - min_v + 1e-12)

valid_data_long["anom_score"] = anom_score

loan_scores = valid_data_long.groupby("index")["anom_score"].mean().reset_index()
loan_scores = loan_scores.merge(Y_valid, on="index", how="left")

ap = average_precision_score(loan_scores["target"], loan_scores["anom_score"])
roc_auc = roc_auc_score(loan_scores["target"], loan_scores["anom_score"])

print(f"AP: {ap:.4f}")
print(f"AUC-ROC : {roc_auc:.4f}")

AP: 0.1333
AUC-ROC : 0.5289


In [60]:
from sklearn.neighbors import LocalOutlierFactor

lof_pipe = Pipeline([
    ("prep", preprocess), ("clf", LocalOutlierFactor(n_neighbors=10, novelty=True, contamination = "auto"))
])

# Fit
lof_pipe.fit(X_train)


,steps,"[('prep', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,1.0
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [61]:
X_valid_scaled = lof_pipe["prep"].transform(X_valid)
scores = lof_pipe["clf"].score_samples(X_valid_scaled)
raw_anom = -scores

min_v, max_v = np.min(raw_anom), np.max(raw_anom)
anom_score = (raw_anom - min_v) / (max_v - min_v + 1e-12)

valid_data_long["anom_score"] = anom_score

loan_scores = valid_data_long.groupby("index")["anom_score"].mean().reset_index()
loan_scores = loan_scores.merge(Y_valid, on="index", how="left")

ap = average_precision_score(loan_scores["target"], loan_scores["anom_score"])
roc_auc = roc_auc_score(loan_scores["target"], loan_scores["anom_score"])

print(f"AP: {ap:.4f}")
print(f"AUC-ROC : {roc_auc:.4f}")

AP: 0.1726
AUC-ROC : 0.5563


In [56]:
from sklearn.svm import OneClassSVM

ocsvm_pipe = Pipeline([
    ("prep", preprocess),
    ("clf", OneClassSVM(kernel="rbf", nu=0.05, gamma="scale"))
])

ocsvm_pipe.fit(X_train)

,steps,"[('prep', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,1.0
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [57]:
X_valid_scaled = ocsvm_pipe["prep"].transform(X_valid)
scores = ocsvm_pipe["clf"].score_samples(X_valid_scaled)
raw_anom = -scores

min_v, max_v = np.min(raw_anom), np.max(raw_anom)
anom_score = (raw_anom - min_v) / (max_v - min_v + 1e-12)

valid_data_long["anom_score"] = anom_score

loan_scores = valid_data_long.groupby("index")["anom_score"].mean().reset_index()
loan_scores = loan_scores.merge(Y_valid, on="index", how="left")

ap = average_precision_score(loan_scores["target"], loan_scores["anom_score"])
roc_auc = roc_auc_score(loan_scores["target"], loan_scores["anom_score"])

print(f"AP: {ap:.4f}")
print(f"AUC-ROC : {roc_auc:.4f}")

AP: 0.1685
AUC-ROC : 0.5776
